In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("=" * 70)
print("FINANCIAL NEWS CLEANING")
print("=" * 70)
print("Reading table: financial_news_raw")

StatementMeta(, d398623f-2801-49f6-a442-a7340c8d4377, 3, Finished, Available, Finished, False)

FINANCIAL NEWS CLEANING
Reading table: financial_news_raw


In [2]:
raw_news_df = spark.table("financial_news_raw")

print("Raw rows:", raw_news_df.count())

display(raw_news_df.limit(10))

StatementMeta(, d398623f-2801-49f6-a442-a7340c8d4377, 4, Finished, Available, Finished, False)

Raw rows: 1835


SynapseWidget(Synapse.DataFrame, df8731ce-875b-40b7-9148-65d3b10ea0eb)

In [3]:
raw_news_df.printSchema()

StatementMeta(, d398623f-2801-49f6-a442-a7340c8d4377, 5, Finished, Available, Finished, False)

root
 |-- datetime: timestamp (nullable = true)
 |-- Ticker: string (nullable = true)
 |-- headline: string (nullable = true)
 |-- summary: string (nullable = true)
 |-- source: string (nullable = true)
 |-- url: string (nullable = true)
 |-- image: string (nullable = true)
 |-- category: string (nullable = true)
 |-- id: long (nullable = true)



In [4]:
clean_news_df = (
    raw_news_df

    # Standardize ticker values
    .withColumn(
        "Ticker",
        F.upper(F.trim(F.col("Ticker")))
    )

    # Clean text columns
    .withColumn(
        "headline",
        F.trim(F.regexp_replace(F.col("headline"), r"\s+", " "))
    )
    .withColumn(
        "summary",
        F.trim(F.regexp_replace(F.col("summary"), r"\s+", " "))
    )
    .withColumn(
        "source",
        F.trim(F.col("source"))
    )

    # Create useful date columns
    .withColumn(
        "Published_Date",
        F.to_date(F.col("datetime"))
    )
    .withColumn(
        "Published_Year",
        F.year(F.col("datetime"))
    )
    .withColumn(
        "Published_Month",
        F.month(F.col("datetime"))
    )
    .withColumn(
        "Published_Hour",
        F.hour(F.col("datetime"))
    )

    # Record when Fabric processed the row
    .withColumn(
        "Processed_At",
        F.current_timestamp()
    )

    # Remove rows without essential data
    .filter(
        F.col("id").isNotNull()
        & F.col("Ticker").isNotNull()
        & F.col("headline").isNotNull()
        & F.col("datetime").isNotNull()
    )

    # Remove duplicates
    .dropDuplicates(["id"])
)

StatementMeta(, d398623f-2801-49f6-a442-a7340c8d4377, 6, Finished, Available, Finished, False)

In [5]:
clean_news_df = clean_news_df.select(
    F.col("id").alias("Article_ID"),
    F.col("Ticker"),
    F.col("datetime").alias("Published_At"),
    F.col("Published_Date"),
    F.col("Published_Year"),
    F.col("Published_Month"),
    F.col("Published_Hour"),
    F.col("source").alias("Source"),
    F.col("headline").alias("Headline"),
    F.col("summary").alias("Summary"),
    F.col("url").alias("Article_URL"),
    F.col("image").alias("Image_URL"),
    F.col("category").alias("Category"),
    F.col("Processed_At")
)

print("Clean rows:", clean_news_df.count())

display(clean_news_df.limit(10))

StatementMeta(, d398623f-2801-49f6-a442-a7340c8d4377, 7, Finished, Available, Finished, False)

Clean rows: 1835


SynapseWidget(Synapse.DataFrame, 478eb48d-2ff7-4e6a-ad35-da0bfc4d08e4)

In [6]:
(
    clean_news_df
        .write
        .mode("overwrite")
        .format("delta")
        .saveAsTable("financial_news_clean")
)

print("Table financial_news_clean saved successfully.")

StatementMeta(, d398623f-2801-49f6-a442-a7340c8d4377, 8, Finished, Available, Finished, False)

Table financial_news_clean saved successfully.


In [7]:
clean_table_df = spark.table("financial_news_clean")

print("Rows stored:", clean_table_df.count())

display(clean_table_df.limit(20))

StatementMeta(, d398623f-2801-49f6-a442-a7340c8d4377, 9, Finished, Available, Finished, False)

Rows stored: 1835


SynapseWidget(Synapse.DataFrame, f1786622-0e1d-44b2-83df-9ce4749e7ba3)

In [8]:
display(
    spark.sql("""
        SELECT
            COUNT(*) AS Total_Rows,
            COUNT(DISTINCT Article_ID) AS Unique_Articles,
            COUNT(DISTINCT Ticker) AS Tickers,
            MIN(Published_At) AS Oldest_Article,
            MAX(Published_At) AS Latest_Article
        FROM financial_news_clean
    """)
)

StatementMeta(, d398623f-2801-49f6-a442-a7340c8d4377, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 170d9884-0499-4dd3-8644-6950da7480fe)

In [9]:
display(
    spark.sql("""
        SELECT
            SUM(CASE WHEN Article_ID IS NULL THEN 1 ELSE 0 END) AS Missing_ID,
            SUM(CASE WHEN Ticker IS NULL OR TRIM(Ticker) = '' THEN 1 ELSE 0 END) AS Missing_Ticker,
            SUM(CASE WHEN Headline IS NULL OR TRIM(Headline) = '' THEN 1 ELSE 0 END) AS Missing_Headline,
            SUM(CASE WHEN Published_At IS NULL THEN 1 ELSE 0 END) AS Missing_Date
        FROM financial_news_clean
    """)
)

StatementMeta(, d398623f-2801-49f6-a442-a7340c8d4377, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e5176600-63cf-489d-b64a-14d8ab4b0f14)

In [10]:
display(
    spark.sql("""
        SELECT
            Ticker,
            COUNT(*) AS Article_Count,
            ROUND(
                COUNT(*) * 100.0 /
                SUM(COUNT(*)) OVER (),
                2
            ) AS Percentage_Of_All_Articles
        FROM financial_news_clean
        GROUP BY Ticker
        ORDER BY Article_Count DESC
    """)
)

StatementMeta(, d398623f-2801-49f6-a442-a7340c8d4377, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0d2030f8-40cb-4a84-8f2c-da032c9d44c0)

In [11]:
clean_count = spark.table("financial_news_clean").count()

print("=" * 70)
print("NEWS CLEANING COMPLETED SUCCESSFULLY")
print("=" * 70)
print(f"Rows stored: {clean_count}")
print("Input table: financial_news_raw")
print("Output table: financial_news_clean")
print("Lakehouse: AI_Financial_Lakehouse")

StatementMeta(, d398623f-2801-49f6-a442-a7340c8d4377, 13, Finished, Available, Finished, False)

NEWS CLEANING COMPLETED SUCCESSFULLY
Rows stored: 1835
Input table: financial_news_raw
Output table: financial_news_clean
Lakehouse: AI_Financial_Lakehouse
